In [1]:
from typing import Optional, List, Type
from pydantic import BaseModel, ValidationError, Field, validator
import pandas as pd
import geopandas as gpd
from shapely import wkb, Point
import numpy as np

**Define data reading and validation function**

In [2]:
# Define  Pydantic model
# Metro Fire Facilities
class Record(BaseModel):
    X: Optional[float] = Field(None, description="X coordinate")
    Y: Optional[float] = Field(None, description="Y coordinate")
    gnaf_confidence: Optional[float] = Field(None, ge=0, le=1, description="GNAF confidence score (0-1)")
    distance_to_gnaf: Optional[float] = Field(None, ge=0, description="Distance to GNAF point in meters")
    gnaf_lat: Optional[float] = Field(None, ge=-90, le=90, description="GNAF latitude")
    gnaf_long: Optional[float] = Field(None, ge=-180, le=180, description="GNAF longitude")
    geom: Optional[str] = Field(None, description="Geometry in WKT format")
    bpa_areaha: Optional[float] = Field(None, ge=0, description="Burned area in hectares")
    allcount: Optional[int] = Field(None, ge=0, description="Total count")
    yrsfrburn: Optional[int] = Field(None, ge=0, description="Years since last burn")
    firecount: Optional[int] = Field(None, ge=0, description="Fire count")
    burncount: Optional[int] = Field(None, ge=0, description="Burn count")
    zonetype: Optional[float] = Field(None, description="Zone type code")



def read_select(path: str, model_class: Type[BaseModel], usecols=None, filters=None, xy_cols=None, wkb_col=None, crs="EPSG:4326", encoding=None):
    encodings_to_try = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']
    if encoding:
        encodings_to_try = [encoding] + encodings_to_try

    df = None
    for enc in encodings_to_try:
        try:
            df = pd.read_csv(path, usecols=usecols + (list(filters.keys()) if filters else []), encoding=encoding)
            break
        except UnicodeDecodeError:
            continue

    if df is None:
        raise ValueError(f"Cannot read file: {path}")


    # Apply filters (on categorical fields)
    if filters:
        for col, val in filters.items():
            if col == "address":  # Use "contains"
                df = df[df[col].str.contains(val, case=False, na=False)]  #Case-insensitive
            else:  # Use exact matching
                df = df[df[col] == val]
        # # Drop filter columns after use
        df = df.drop(columns=list(filters.keys()), errors="ignore")

    df = df.dropna(subset=usecols)

    # Convert to Pydantic model
    records: List[BaseModel] = []
    for row in df.to_dict(orient="records"):
        filtered_row = {k: v for k, v in row.items() if k in model_class.model_fields}
        try:
            records.append(model_class(**filtered_row))
        except Exception as e:
            pass

    # Convert to GeoDataFrame if needed
    if xy_cols:
        gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[xy_cols[0]], df[xy_cols[1]]), crs=crs)
    elif wkb_col:
        df["geometry"] = df[wkb_col].apply(lambda x: wkb.loads(bytes.fromhex(x), hex=True))
        gdf = gpd.GeoDataFrame(df.drop(columns=[wkb_col]), geometry="geometry", crs=crs)
    else:
        gdf = df  # just return the DataFrame if no geometry


    # keep columns
    columns = [c for c in df.columns if c != "geometry"]

    return records, gdf, columns

**Data readling and validation**

In [3]:
# Metro Fire Facilities
metro_records, metro_gdf,  metro_cols = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/Metro_Fire_Facilities.csv",
    model_class=Record,
    filters={"facility_operationalstatus": "OPERATIONAL", "facility_state": "VICTORIA"},
    usecols=["X","Y","gnaf_confidence","distance_to_gnaf","gnaf_lat","gnaf_long"],
    xy_cols=("X","Y")
)

In [4]:
rural_records, rural_gdf, rural_cols= read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/Rural_Country_Fire_Service_Facilities.csv",
    model_class=Record,
    filters={"facility_operationalstatus": "OPERATIONAL", "facility_state": "VICTORIA"},
    usecols=["X","Y","gnaf_confidence","distance_to_gnaf","gnaf_lat","gnaf_long"],
    xy_cols=("X","Y")
)

In [5]:
bushfire_records, bushfire_gdf,bushfire_cols = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/vic_fire_bushfire_prone_area.csv",
    model_class=Record,
    usecols=["geom","bpa_areaha"],
    wkb_col="geom"
)

In [6]:
fire_history_records, fire_history_gdf, fire_history_cols  = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/vic_fire_history_latest.csv",
    model_class=Record,
    usecols=["geom","allcount","yrsfrburn","firecount","burncount"],
    wkb_col="geom"
)

In [7]:
fire_manage_records, fire_manage_gdf,fire_manage_cols = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/vic_fire_manage_zone.csv",
    model_class=Record,
    usecols=["geom","zonetype"],
    wkb_col="geom"
)

In [8]:
renewable_records, renewable_gdf, renewable_cols = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/renewable_project_nem.csv",
    model_class=Record,
    filters={"Region":"VIC1"},
    usecols=["X","Y"],
    xy_cols=("X","Y"),
    encoding='latin-1'
)

In [9]:
transmission_station_records, transmission_station_gdf, transmission_station_cols = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/transmission_substation.csv",
    model_class=Record,
    filters={"operationa": "Operational"},
    usecols=["geom","address"],
    wkb_col="geom"
)

In [10]:
property_records, property_gdf,property_cols = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/vic_fire_manage_zone.csv",
    model_class=Record,
    usecols=["geom"],
    wkb_col="geom"
)

**Training Data creation**

In [11]:
# first column is x
# second column is y
# third column is radius
# fouth column is total number of facilites
# fifth column is the closest facility

train_data=[]
id=0
#for i in range(0,len(transmission_station_records)):
for i in range(0,1000):


  # for j in range(0,len(property_records)):
  for j in range(0,100):


    # determine the property block location
    x = property_gdf.iloc[j].geometry.centroid.x
    y = property_gdf.iloc[j].geometry.centroid.y
    location = np.array([x, y])

    # calculate the distance between transmission station and property block
    station_location = np.array([transmission_station_gdf.iloc[i].geometry.x, transmission_station_gdf.iloc[i].geometry.y])
    distance = np.linalg.norm(location - station_location)

    # if the property block locate within the station in 10km (1 degree distance is around 100km)
    # then means this property block is within 10km radius area of the transmission station
    if distance < 0.1:
      id=id+1
      # use the area of 5km radius within the center of property block as property block area
      radius=0.05

      # define the closest facility distance to current property block
      closest_distance=10000

      # define the total number of facility within current property block area
      number=0

      # search metro fire facilities that is within the area
      for k in range(0,10):
        facility = np.array([metro_gdf.iloc[k].geometry.x, metro_gdf.iloc[k].geometry.y])
        distance = np.linalg.norm(location - facility)

        if distance < radius:
            number+=1

        if distance < closest_distance:
            closest_distance = distance

      # search rural fire facilities that is within the area
      for k in range(0,10):
        facility = np.array([rural_gdf.iloc[k].geometry.x, rural_gdf.iloc[k].geometry.y])
        distance = np.linalg.norm(location - facility)

        if distance < radius:
            number+=1

        if distance < closest_distance:
            closest_distance = distance

      # check whether the area is a bushfire prone area
      bushfire_area=0
      for k in range(0,10):
        if Point(location).within(bushfire_gdf.iloc[k].geometry):
          bushfire_area=1

      # for k in range(0,10):
      #   if Point(location).within(fire_manage_gdf.iloc[k].geometry):
      #     fire_zone_type=fire_manage_gdf.iloc[k].zonetype


      # count th fire and burn counts within the area
      fire_count_area=0
      burn_count_area=0
      yrsfrburn_area=0
      for k in range(0,10):
        if Point(location).within(fire_history_gdf.iloc[k].geometry):
          fire_count_area=fire_history_gdf.iloc[k].firecount
          burn_count_area=fire_history_gdf.iloc[k].burncount
          yrsfrburn_area=fire_history_gdf.iloc[k].yrsfrburn

     #training vector
      row = [id, x, y, radius, number, closest_distance,bushfire_area,fire_count_area,burn_count_area,yrsfrburn_area]
      train_data.append(row)

# convert traning vector to numpy
train_data=np.array(train_data)

**label generation**

In [ ]:
def calculate_risk_level(record,
                         facility_count,
                         closest_distance,
                         weight_fire_history=0.3,
                         weight_facility=0.4,
                         weight_zone=0.3):
    # --------------------------
    # 1. Fire History Score (0-100 points)
    # This factor comprehensively considers the occurrence of fires (fire count) and burns (burn count),
    # and applies a decay correction based on the time since the last fire.
    # --------------------------
    fire_history_score = 0

    if record.allcount and record.allcount > 0:
        # 1.1 Calculate the total proportion of fire and incineration incidents
        total_incidents = (record.firecount or 0) + (record.burncount or 0)
        incident_ratio = min(total_incidents / record.allcount, 1.0)

        # 1.2 Distinguishing the weighted contributions of fire and incineration (fire accounts for 60%, incineration accounts for 40%)
        if total_incidents > 0:
            fire_contribution = (record.firecount or 0) / total_incidents * 60
            burn_contribution = (record.burncount or 0) / total_incidents * 40
        else:
            fire_contribution = 0
            burn_contribution = 0
        base_score = (fire_contribution + burn_contribution) * incident_ratio

        # 1.3 Time factor correction: The closer the time to the last fire, the higher the risk. The basic score is adjusted by the time factor.
        if record.yrsfrburn is not None:
            if record.yrsfrburn <= 5:
                time_factor = 1.3
            elif record.yrsfrburn <= 10:
                time_factor = 1.1
            elif record.yrsfrburn <= 20:
                time_factor = 0.8
            else:
                time_factor = 0.5
        else:
            time_factor = 1.0

        fire_history_score = min(base_score * time_factor, 100)

    # --------------------------
    # 2. Fire Facility Score (0-100 points)
    # Evaluate the number of fire facilities and the distance to the nearest fire facility, taking the more unfavorable score.
    # --------------------------
    facility_score = 0

    if facility_count is not None:
        # 5 or more firefighting facilities are considered sufficient. The fewer the number, the higher the score (higher the risk).
        count_score = max(0, 100 - (min(facility_count, 5) * 20))
    else:
        count_score = 100

    # 2.2 Scoring based on distance to the nearest firefighting facility: the farther the distance, the higher the risk
    if closest_distance is not None:
        distance_km = closest_distance / 1000
        if distance_km >= 10:
            distance_score = 100
        elif distance_km >= 5:
            distance_score = 80
        elif distance_km >= 2:
            distance_score = 50
        elif distance_km >= 1:
            distance_score = 30
        else:
            distance_score = 10
    else:
        distance_score = 100

    facility_score = (count_score * 0.6) + (distance_score * 0.4)
    facility_score = max(0, min(facility_score, 100))

    # --------------------------
    # 3. Regional Type Factor Score (0-100 points)
    # Scoring is based on the risk level associated with different regional types.
    # --------------------------

    zone_risk_mapping = {
        0: 50,   # Not Zoned：Medium
        1: 30,   # Asset Protection Zone：Low
        2: 60,   # Bushfire Moderation Zone：Medium
        3: 80    # Landscape Management Zone：High
    }
    zone_score = zone_risk_mapping.get(record.zonetype, 50)

    # --------------------------
    # Comprehensive risk calculation (0-100 points)
    # Calculate the final risk level based on the weight of each factor.
    # --------------------------
    total_risk = (
        fire_history_score * weight_fire_history +
        facility_score * weight_facility +
        zone_score * weight_zone
    )

    # Ensure the final risk level is between 0-100 and retain 1 decimal place
    return round(min(max(total_risk, 0), 100), 1)

In [ ]:
risk_levels = []
for i in range(len(transmission_station_records)):
    substation_id = i
    location_record = fire_history_records[i] if i < len(fire_history_records) else Record()
    facility_count = train_data[i][4]
    closest_distance = train_data[i][5]

    risk = calculate_risk_level(
        record=location_record,
        facility_count=facility_count,
        closest_distance=closest_distance
    )

    risk_levels.append({
        "substation_id": substation_id,
        "x": train_data[i][1],
        "y": train_data[i][2],
        "risk_level": risk
    })

In [ ]:
print(risk_levels[1:20])

[{'substation_id': 1, 'x': np.float64(145.52972082000008), 'y': np.float64(-33.515932515999964), 'risk_level': 31.0}, {'substation_id': 2, 'x': np.float64(144.87828941600003), 'y': np.float64(-34.48941824299993), 'risk_level': 25.6}, {'substation_id': 3, 'x': np.float64(149.3981758530001), 'y': np.float64(-34.79553015999994), 'risk_level': np.float64(30.4)}, {'substation_id': 4, 'x': np.float64(149.3831752110001), 'y': np.float64(-34.695830899999976), 'risk_level': np.float64(30.4)}, {'substation_id': 5, 'x': np.float64(151.70367033200012), 'y': np.float64(-32.67054972899996), 'risk_level': np.float64(49.6)}, {'substation_id': 6, 'x': np.float64(138.76929655100002), 'y': np.float64(-35.171803764999936), 'risk_level': np.float64(45.4)}, {'substation_id': 7, 'x': np.float64(138.81996439500006), 'y': np.float64(-34.99688287199996), 'risk_level': np.float64(45.4)}, {'substation_id': 8, 'x': np.float64(138.812985098), 'y': np.float64(-35.04358029499997), 'risk_level': np.float64(45.4)}, {'s